# N120 · 回文DP、中心扩展与Manacher

**目标：** 从对称复用推导线性回文半径算法。

**先修：** N098, N118。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 奇偶中心；回文区间DP；镜像中心；右边界；分隔符风险。

**配套讲解来源：** [同名逐章意见](../../comment/120_palindrome_algorithms_manacher.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这章解决两个经典问题，并给出三档速度越来越快的解法。

问题一（最长回文子串）：给你一个字符串 s，你要找出它里面最长的回文子串。回文就是正着读、倒着读都一样的串，例如 "aba"、"abba"。具体例子：输入 s = "babad"，答案是 "bab" 或 "aba"（两个都合法，长度都是 3）。为什么是它？我们把子串从长到短检查：长度 4 的 "baba"、"abad" 反过来都不等于自己；长度 3 的 "bab" 反过来仍是 "bab"，所以最长回文的长度就是 3。

问题二（回文子串计数）：你要数出 s 里一共有多少个回文子串，注意是按出现位置计数、不去重。具体例子：输入 s = "aaa"，答案是 6：三个 "a"（下标 0、1、2 各算一个）、两个 "aa"（下标 0–1、1–2 各算一个）、一个 "aaa"，加起来 3+2+1 = 6。

暴力做法是枚举全部 O(n²) 个子串、每个花 O(n) 反转比较，总共 O(n³)。本章给出三档解法：区间 DP（O(n²)）、中心扩展（O(n²) 但省空间）、Manacher（O(n)）。三档解法层层递进，最终的看点是 Manacher 如何把已经比较过的信息"白嫖"回来。

## 2. 基础知识：先读懂这些词

- **回文子串**：字符串里连续的一段，正读反读完全一样。注意我们要的是"子串"（必须连续），不是可以跳着挑的"子序列"。
- **奇中心 / 偶中心**：回文的对称轴有两种落位。像 "aba" 的轴压在字符 'b' 身上，回文长度是奇数；像 "abba" 的轴落在两个 'b' 中间，回文长度是偶数。两种轴必须分开枚举，这是贯穿全章的主题。
- **回文半径**：以某个中心为轴，回文向两边延伸的层数。本章代码约定：odd[i] 是奇中心 i 的半径且包含中心字符本身（odd[i]=3 对应长度 2×3−1=5 的回文）；even[i] 是偶中心的"配对数"（even[i]=2 对应长度 2×2=4 的回文）。
- **区间 DP**：dp[l][r] 是一张 n×n 的布尔表，dp[l][r]=True 表示 s[l..r] 这一段是回文。它依赖的事实是：s[l..r] 是回文，当且仅当首尾字符相同、且掐头去尾后的内部段 s[l+1..r−1] 也是回文。
- **中心扩展**：枚举每一条可能的对称轴，从轴向两边一格一格比较，比不动为止。它比区间 DP 省内存（只需 O(1) 额外空间）。
- **Manacher（马拉车）**：线性时间求出每个中心的回文半径。它的关键动作是"镜像复用"：如果当前中心 i 落在一个已经确认的大回文内部，那么 i 的回文至少和它镜像点的回文一样长（或延伸到大回文右边界为止），这些比较就不用重做了。
- **镜像中心**：设已知大回文区间是 [left, right]，当前中心 i 在它内部，则 i 的镜像点是 left+right−i（把 i 关于区间中点翻转）。大回文左右完全对称，所以镜像点能延伸多远，i 在区间内部就能延伸多远。
- **右边界（left, right）**：Manacher 始终维护"到目前为止见过的、右端点最靠右的那个回文区间"。这个右边界是算法能做到 O(n) 的发动机：每一次"真正"的新比较都会把右边界往右推。
- **记忆化 / 复用**：这里体现的思想和记忆化搜索同源——算过的答案存起来（存在 odd 数组和 left/right 里），后面遇到能直接用就不再重算。
- **分隔符风险**：很多教材把字符串插 '#' 变成 "#a#b#a#" 来统一奇偶中心，但若输入串本身可能含 '#'，这个技巧就会出错。本章选择不插分隔符、奇偶两轮分开算，从根上避开这个坑。

## 3. 思路推导：从小例子开始

我们从最直观的中心扩展出发，一步步逼出 Manacher。

- Step 1：把"找最长回文"改写成"枚举对称轴"。任何回文都有一条对称轴，轴要么压在字符上（奇），要么卡在两个字符之间（偶），总共只有 2n−1 个候选轴。我们逐条轴扩张，就能得到以每条轴为中心的最长回文，再取全局最长。
- Step 2：先手算一遍中心扩展。取 s = "abba"，考察偶轴（l=1, r=2，即两个 'b' 中间）：s[1]='b' 等于 s[2]='b'，匹配，扩到 l=0, r=3；s[0]='a' 等于 s[3]='a'，再匹配，扩到 l=−1, r=4，越界停。最终回文区间是 [0,3]，答案 "abba"。
- Step 3：中心扩展慢在哪？它慢在每换一条轴，之前比较过的信息全部丢弃、从零再比一遍。Manacher 的思路就是把这些信息留住：我记住房号区间 [left, right]（目前右端最远的回文）。新中心 i 如果落在 [left, right] 里面，它的半径可以先用镜像点的结果"预支"一部分，预支范围内的比较直接跳过。
- Step 4：预支多少要分两种情况。镜像点 j = left+right−i 的半径是 odd[j]：如果 odd[j] 比较小、整段都落在大回文内部，那么 i 的半径直接等于 odd[j]，while 循环第一次比较就会失败，一步都不用白比；如果 odd[j] 顶到或超出右边界，那么大回文之外的字符是未知的，我们只敢保证到右边界，radius 从 right−i+1 起步，后面的字符必须老老实实逐个比较。
- Step 5：手算 Manacher 的奇半径轮。s = "abacabba"（下标 0–7：a b a c a b b a）。下表就是 notebook 里 “运行示例”部分 实际生成的表：

| 中心下标 i | 奇半径(含中心) odd[i] | 偶半径 even[i] | 最长奇回文 | 最长偶回文 |
|---|---|---|---|---|
| 0 | 1 | 0 | a | （空） |
| 1 | 2 | 0 | aba | （空） |
| 2 | 1 | 0 | a | （空） |
| 3 | 3 | 0 | bacab | （空） |
| 4 | 1 | 0 | a | （空） |
| 5 | 1 | 0 | a | （空） |
| 6 | 1 | 2 | b | abba |
| 7 | 1 | 0 | a | （空） |

我们挑几个关键瞬间看：
  - i=1：i 在右边界之外（right 还是 0），没得白嫖，radius 从 1 起步硬比。s[0]='a' 等于 s[2]='a'，半径涨到 2；再扩就越界了。于是 odd[1]=2，对应 "aba"，同时右边界更新为 [0,2]。
  - i=3：i=3 大于 right=2，仍在已知窗口外，radius 从 1 硬比：s[2]='a' 等于 s[4]='a'，半径 2；s[1]='b' 等于 s[5]='b'，半径 3；s[0]='a' 不等于 s[6]='b'，停。odd[3]=3，对应 "bacab"，右边界推进到 [1,5]。
  - i=5：i=5 正好落在窗口 [1,5] 内！镜像点是 1+5−5=1，odd[1]=2，但 2 已经顶到右边界（2 ≥ right−i+1 = 1），被截断成 radius=1 起步，然后硬比 s[4]='a' 与 s[6]='b' 失败，odd[5]=1。这就是"镜像半径被右边界截断、必须实比"的典型情形。
- Step 6：偶半径轮照同样规则再跑一遍，只是轴的位置换到 i−1 与 i 之间，radius=0 表示紧挨着的这对字符不相等。i=6 时：s[5]='b' 等于 s[6]='b'，radius 涨到 1；s[4]='a' 等于 s[7]='a'，涨到 2；再扩越界停。even[6]=2，对应 "abba"。
- Step 7：计数。以奇中心 i 为轴，半径取 1、2、…、odd[i] 的每一层都是一个回文，所以这个中心恰好贡献 odd[i] 个；偶中心同理贡献 even[i] 个。对 "abacabba"：奇半径合计 1+2+1+3+1+1+1+1 = 11，偶半径合计 2，总共 13 个回文子串。你可以自己数一遍验证：8 个单字符，加 "aba"、"aca"、"bacab"，加 "bb"、"abba"，正好 13 个。

## 4. 核心代码：longest_palindrome_center

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def longest_palindrome_center(s):
    best = (0, 0)
    for center in range(len(s)):
        for l, r in [(center, center), (center, center + 1)]:
            while l >= 0 and r < len(s) and (s[l] == s[r]):
                l -= 1
                r += 1
            if r - l - 1 > best[1] - best[0]:
                best = (l + 1, r)
    return s[best[0]:best[1]]
```

### 逐段读懂代码

### 4.1 longest_palindrome_center：中心扩展求最长回文子串

```python
def longest_palindrome_center(s):
    best = (0, 0)
    for center in range(len(s)):
        for l, r in [(center, center), (center, center + 1)]:
            while l >= 0 and r < len(s) and (s[l] == s[r]):
                l -= 1
                r += 1
            if r - l - 1 > best[1] - best[0]:
                best = (l + 1, r)
    return s[best[0]:best[1]]
```

- `best=(0,0)` 表示当前最优解是空切片 s[0:0]，即空串。它保证空串输入也能安全返回 ''。
- `for l,r in [(center,center),(center,center+1)]` 是一次枚举两条轴：第一对是奇轴（l、r 重合在一个字符上），第二对是偶轴（l、r 是紧挨着的两个字符）。
- `while l>=0 and r<len(s) and s[l]==s[r]: l-=1; r+=1` 这一行是紧凑写法：循环体"左指针减一、右指针加一"被分号挤在同一行。循环停下来时，l 和 r 都已经越过了回文的边界（要么越界、要么字符不等），所以真正的回文区间是 [l+1, r−1]，长度是 r−l−1。
- `if r-l-1>best[1]-best[0]` 比较的是"当前回文长度 r−l−1"与"当前最优切片的长度 best[1]−best[0]"。若更长，就把 `best` 更新成 `(l+1,r)`，即回文的切片区间。
- 最后 `return s[best[0]:best[1]]` 直接切出答案。

### 4.2 palindrome_dp：区间 DP 表

```python
def palindrome_dp(s):
    n = len(s)
    dp = [[False] * n for _ in range(n)]
    for l in range(n - 1, -1, -1):
        for r in range(l, n):
            dp[l][r] = s[l] == s[r] and (r - l < 2 or dp[l + 1][r - 1])
    return dp
```

- 这个函数返回整张 dp 表（不是子串），主要供测试交叉验证用。
- `for l in range(n-1,-1,-1)` 让左端点 l 从右往左算。这样当你算 dp[l][r] 时，它依赖的 dp[l+1][r−1] 属于更靠右的行，早就算好了。
- `r-l<2 or dp[l+1][r-1]` 处理长度 1 和 2 的段：长度 1 或 2 时内部没有字符（或说内部是空），不用（也不能）去查内部格子，只要两端相等就成立；更长的段则要求"两端相等且内部也是回文"。
- Python 的 `and` 短路求值在这里顺便当了效率优化：两端字符已经不同时，根本不会去碰 dp[l+1][r-1]。

### 4.3 manacher_radii：Manacher 求每个中心的半径

（对应程序见下方分段实现与完整代码。）

- 奇数轮：`left=0; right=-1` 表示初始窗口为空（还没有任何已知回文）。`radius=1 if i>right else min(odd[left+right-i],right-i+1)` 是紧凑的三目表达式：i 在窗口外时没有信息可用，radius 从 1（只有中心字符）起步；i 在窗口内时，radius 直接预支镜像点 left+right−i 的半径，但用 right−i+1 封顶（不许越过右边界）。
- `while i-radius>=0 and i+radius<n and s[i-radius]==s[i+radius]: radius+=1` 做剩下的真实比较：看中心两侧对称位置 i−radius 与 i+radius 的字符是否相等，相等就把半径加一继续扩。
- `if i+radius-1>right: left,right=i-radius+1,i+radius-1` 在新回文的右端点 i+radius−1 更靠右时，更新窗口为这个新回文 [i−radius+1, i+radius−1]。
- 偶数轮的结构一模一样，但有两个"差一"：轴在 i−1 与 i 之间，所以 while 里比较的是 s[i−radius−1] 与 s[i+radius]（radius=0 时就是检查紧挨着的 s[i−1] 和 s[i]）；radius 可以是 0，表示这对字符根本不相等、没有偶回文。镜像点也因此偏移半格，变成 left+right−i+1；偶回文 [i−radius, i+radius−1] 的更新式同样偏了一格。
- 函数返回 (odd, even) 两个数组，语义是：以 i 为奇中心的最长回文长度为 2*odd[i]−1，以 i−1/i 之间为偶中心的最长回文长度为 2*even[i]。

### 4.4 count_palindromic_substrings：半径求和即计数

```python
def count_palindromic_substrings(s):
    odd, even = manacher_radii(s)
    return sum(odd) + sum(even)
```

- 这函数只有一行：先拿到两轮半径数组，再把它们全部加起来。它能这么写的原因见 Step 7——每个中心贡献恰好等于其半径数量的回文。

## 5. 分段实现：按顺序运行

**本章接口：** `longest_palindrome_center(s)`、`palindrome_dp(s)`、`manacher_radii(s)`、`count_palindromic_substrings(s)`

### longest_palindrome_center

In [1]:
def longest_palindrome_center(s):
    best = (0, 0)
    for center in range(len(s)):
        for l, r in [(center, center), (center, center + 1)]:
            while l >= 0 and r < len(s) and (s[l] == s[r]):
                l -= 1
                r += 1
            if r - l - 1 > best[1] - best[0]:
                best = (l + 1, r)
    return s[best[0]:best[1]]

### palindrome_dp

In [2]:
def palindrome_dp(s):
    n = len(s)
    dp = [[False] * n for _ in range(n)]
    for l in range(n - 1, -1, -1):
        for r in range(l, n):
            dp[l][r] = s[l] == s[r] and (r - l < 2 or dp[l + 1][r - 1])
    return dp

### manacher_radii

In [3]:
def manacher_radii(s):
    n = len(s)
    odd = [0] * n
    left = 0
    right = -1
    for i in range(n):
        radius = 1 if i > right else min(odd[left + right - i], right - i + 1)
        while i - radius >= 0 and i + radius < n and (s[i - radius] == s[i + radius]):
            radius += 1
        odd[i] = radius
        if i + radius - 1 > right:
            left, right = (i - radius + 1, i + radius - 1)
    even = [0] * n
    left = 0
    right = -1
    for i in range(n):
        radius = 0 if i > right else min(even[left + right - i + 1], right - i + 1)
        while i - radius - 1 >= 0 and i + radius < n and (s[i - radius - 1] == s[i + radius]):
            radius += 1
        even[i] = radius
        if i + radius - 1 > right:
            left, right = (i - radius, i + radius - 1)
    return (odd, even)

### count_palindromic_substrings

In [4]:
def count_palindromic_substrings(s):
    odd, even = manacher_radii(s)
    return sum(odd) + sum(even)

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

s='abacabba'; odd,even=manacher_radii(s)
show_table(['中心下标','奇半径(含中心)','偶半径(中心在i前)','最长奇回文','最长偶回文'],[(i,odd[i],even[i],s[i-odd[i]+1:i+odd[i]],s[i-even[i]:i+even[i]]) for i in range(len(s))])

中心下标,奇半径(含中心),偶半径(中心在i前),最长奇回文,最长偶回文
0,1,0,a,
1,2,0,aba,
2,1,0,a,
3,3,0,bacab,
4,1,0,a,
5,1,0,b,
6,1,2,b,abba
7,1,0,a,


## 7. 正确性、适用条件与复杂度

镜像回文在当前大回文右界以内的部分由对称性保证，超过右界的字符尚未知，必须显式比较。每次真正扩展使全局最右边界前进，累计O(n)。每个奇中心半径r贡献r个非空回文，每个偶中心同理，所以求和得到按位置计数而非去重字符串数。

**代价：** 中心扩展O(n²)时间O(1)辅助；区间DP O(n²)时间空间；Manacher O(n)时间空间。odd[i]包含中心字符，even[i]以i−1与i之间为中心。

### 展开理解

正确性我们分三个算法说。

中心扩展为什么对：一条轴从中心向外扩张，扩到第一次不匹配为止，得到的必然是以这条轴为中心的最长回文。所有可能的轴都被枚举了一遍，全局最优一定藏在某条轴的结果里，取最大即可。

区间 DP 为什么对：一个串是回文，等价于"首尾相同，且掐头去尾后剩下的内部串也是回文"。这句话对长度 1、2 的串自动成立（它们没有内部）。代码按 l 从大到小填表，保证用到 dp[l+1][r−1] 时那一格已经填好，不会引用到未初始化的值。

Manacher 为什么对：关键在镜像复用是安全的。第一种情形，镜像点的回文完整地躺在大回文 [left, right] 内部：因为大回文左右对称，把镜像点那段的形状翻过来就是 i 的形状，所以 i 的半径直接照抄不会多报；这时 while 的第一次真实比较必然失败，一点不亏。第二种情形，镜像点的回文顶到或超出右边界：右边界之外的字符我们从没比较过，谁也不知道它们等不等，所以只敢保证到右边界（radius 取 right−i+1 起步），剩下的必须真比，这也不会少报。两种情形合起来，初值既不虚高、又把能省的比较都省了。

复杂度（拿具体 n 感受一下）：

- 中心扩展：2n 条轴、每条最多扩 O(n)，总时间 O(n²)、额外空间 O(1)。n=1000 时大约百万次比较，很轻松；n=10⁵ 时约百亿次，等不到结果。
- 区间 DP：填 n×n 的表，时间空间都是 O(n²)。n=1000 是一百万格，没问题；n=10⁵ 是一百亿格，内存先爆。
- Manacher：每次成功的字符比较都会让当前回文的右端 i+radius−1 超过旧的 right，也就是说每次成功的比较都把 right 往右推至少一格；right 从 −1 出发、最多推到 n−1，所以全程成功的比较加起来不超过 n 次，另外每个中心最多再搭一次失败比较。总比较次数是 O(n)。n=10⁵ 时也就十万次上下的比较，一瞬间算完。

计数口径提醒：本章的计数按"位置"数（"aaa" 里两个 "aa" 算两个），不是去重后的不同字符串数。

## 8. 单元测试：每个用例在检查什么

```python
assert longest_palindrome_center('babad') in ('bab', 'aba')
```
这条测正常输入，并且专门照顾"并列第一"：'babad' 里 "bab" 和 "aba" 都是长度 3 的最优解，代码用 `in` 接受两者之一，说明作者清楚哪个先被记录取决于枚举顺序。

```python
assert longest_palindrome_center('cbbd') == 'bb' and count_palindromic_substrings('aaa') == 6
```
前半条专测偶中心：'cbbd' 里唯一像样的回文是 "bb"，如果你的代码忘了枚举 (center, center+1) 这条偶轴，就会返回单字符而挂掉。后半条测计数：'aaa' 的回文个数 3+2+1=6，是奇偶回文都密集的极端样本。

```python
assert longest_palindrome_center('') == '' and manacher_radii('') == ([], [])
```
这两条是边界测试：空串。空串没有中心，循环一次都不进，中心扩展应返回空串，Manacher 应返回两个空数组。

后面那一大段 `for n in range(8): for chars in product('ab',repeat=n)` 是"对拍"（和暴力解互相验证）：它枚举所有长度 0 到 7 的 a/b 字符串（共 255 个），用最笨的方法切出全部子串、逐个反转比较生成标准答案，然后断言三件事——回文总数三算一致（count 函数 == 暴力个数 == dp 表里 True 的格数）；最长长度三算一致（中心扩展结果的长度 == 暴力最长 == 奇偶半径换算出的最大长度）；dp 表每一格都和逐段反转检查的结果一致。小规模全量穷举最能抓"差一错误"，它相当于把本章三种算法交叉锁死。

In [6]:
assert longest_palindrome_center('babad') in ('bab', 'aba')

assert longest_palindrome_center('cbbd') == 'bb' and count_palindromic_substrings('aaa') == 6

assert longest_palindrome_center('') == '' and manacher_radii('') == ([], [])

from itertools import product

for n in range(8):
    for chars in product('ab', repeat=n):
        s = ''.join(chars)
        all_pal = [s[l:r] for l in range(n) for r in range(l + 1, n + 1) if s[l:r] == s[l:r][::-1]]
        odd, even = manacher_radii(s)
        table = palindrome_dp(s)
        assert count_palindromic_substrings(s) == len(all_pal) == sum((sum(row) for row in table))
        longest = max([0] + [len(p) for p in all_pal])
        assert len(longest_palindrome_center(s)) == longest == max([0] + [2 * x - 1 for x in odd] + [2 * x for x in even])
        for l in range(n):
            for r in range(l, n):
                assert table[l][r] == (s[l:r + 1] == s[l:r + 1][::-1])

print('N120: 所有本章断言通过')

N120: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 从半径统计全部回文数。

**练习 2：** 说明分隔符可能出现在输入时如何处理。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（从半径统计全部回文数）**：提示其实就写在 count_palindromic_substrings 里——每个奇中心贡献 odd[i] 个、每个偶中心贡献 even[i] 个，把两个数组求和即可。你要补的是"为什么每个中心恰好贡献半径个数"这句话的论证（半径 1 到 r 的每一层都是一个合法回文，且它们互不相同、都以其为中心）。建议你先拿 "aaa"（答案 6）和 "abba"（答案 6：4 个单字符 + "bb" + "abba"）手算一遍，再写代码对拍。边界别忘空串应得 0。

**练习 2（分隔符出现在输入时怎么办）**：先想清楚病根——插 '#' 技巧的前提是"分隔符绝不出现在原文里"；一旦输入里本身就有 '#'，插完之后你分不清哪个 '#' 是自己插的，镜像复用会把真字符和分隔符错误配对。处理方向有二：要么选一个能保证不在字符集里的哨兵字符（并在接口文档里写死这个前提）；要么像本章实现一样干脆不插分隔符，奇中心、偶中心两轮分开算，对任意字符集都天然安全。你写答案时先给一个含 '#' 的小例子手算演示插分隔符如何出错，再展示你的方案为什么躲开了。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
def longest_palindrome_center(s):
    best = (0, 0)
    for center in range(len(s)):
        for l, r in [(center, center), (center, center + 1)]:
            while l >= 0 and r < len(s) and (s[l] == s[r]):
                l -= 1
                r += 1
            if r - l - 1 > best[1] - best[0]:
                best = (l + 1, r)
    return s[best[0]:best[1]]

def palindrome_dp(s):
    n = len(s)
    dp = [[False] * n for _ in range(n)]
    for l in range(n - 1, -1, -1):
        for r in range(l, n):
            dp[l][r] = s[l] == s[r] and (r - l < 2 or dp[l + 1][r - 1])
    return dp

def manacher_radii(s):
    n = len(s)
    odd = [0] * n
    left = 0
    right = -1
    for i in range(n):
        radius = 1 if i > right else min(odd[left + right - i], right - i + 1)
        while i - radius >= 0 and i + radius < n and (s[i - radius] == s[i + radius]):
            radius += 1
        odd[i] = radius
        if i + radius - 1 > right:
            left, right = (i - radius + 1, i + radius - 1)
    even = [0] * n
    left = 0
    right = -1
    for i in range(n):
        radius = 0 if i > right else min(even[left + right - i + 1], right - i + 1)
        while i - radius - 1 >= 0 and i + radius < n and (s[i - radius - 1] == s[i + radius]):
            radius += 1
        even[i] = radius
        if i + radius - 1 > right:
            left, right = (i - radius, i + radius - 1)
    return (odd, even)

def count_palindromic_substrings(s):
    odd, even = manacher_radii(s)
    return sum(odd) + sum(even)

# 示例与回归测试
assert longest_palindrome_center('babad') in ('bab', 'aba')

assert longest_palindrome_center('cbbd') == 'bb' and count_palindromic_substrings('aaa') == 6

assert longest_palindrome_center('') == '' and manacher_radii('') == ([], [])

from itertools import product

for n in range(8):
    for chars in product('ab', repeat=n):
        s = ''.join(chars)
        all_pal = [s[l:r] for l in range(n) for r in range(l + 1, n + 1) if s[l:r] == s[l:r][::-1]]
        odd, even = manacher_radii(s)
        table = palindrome_dp(s)
        assert count_palindromic_substrings(s) == len(all_pal) == sum((sum(row) for row in table))
        longest = max([0] + [len(p) for p in all_pal])
        assert len(longest_palindrome_center(s)) == longest == max([0] + [2 * x - 1 for x in odd] + [2 * x for x in even])
        for l in range(n):
            for r in range(l, n):
                assert table[l][r] == (s[l:r + 1] == s[l:r + 1][::-1])

print('N120: 所有本章断言通过')

N120: 所有本章断言通过


## 11. 代表题与迁移

- **5. Longest Palindromic Substring**：就是本章问题一的原型题，直接套 longest_palindrome_center，或者用 manacher_radii 找最大半径再切串。这题练的是"枚举对称轴"的建模。
- **647. Palindromic Substrings**：就是本章问题二的原型题，count_palindromic_substrings 一行搞定。这题练的是"半径求和等于按位置计数"这个换算。
- **214. Shortest Palindrome**：要在字符串前面补尽量少的字符使整体成为回文，关键子问题是"从下标 0 出发的最长回文前缀有多长"，正好可以用 Manacher 的半径数组读出来。这题是把本章工具和新目标组合的延伸练习。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。